# Train MiTA-Segmenter to mask faces and license plates

**Task:** per-pixel `background` vs `private` (face ∪ plate) on PP4AV street images, boxes filled in as masks.

**Model:** MiTA-Segmenter: a ViT encoder with MiTA attention + mask-transformer decoder, ImageNet-pretrained encoder.
Model code is imported from the MiTA-Segmenter package; everything that defines the run is in this notebook.

**Choosing the model:** set `backbone` at the top of the config cell, then run the notebook top to bottom. Any non-distilled
ViT in MiTA-Segmenter's `segm/config.yml` works: `vit_tiny_patch16_384`, `vit_small_patch16_384`, `vit_base_patch16_384`,
`vit_large_patch16_384`, `vit_small_patch32_384`, `vit_base_patch32_384`, `vit_large_patch32_384`, `vit_base_patch8_384`.
Everything that depends on it (encoder shape, normalisation, input size, upscale factor, run directory) is derived in section 1b.

**Design (from the EDA):**

| choice | why |
|---|---|
| images upscaled so one model patch covers 8 original pixels (2× for patch-16 backbones) | most faces/plates are under 16 px, smaller than one patch at native resolution |
| half the training crops are centred on an object | face/plate pixels are ~0.25% of the data |
| loss = cross-entropy + Dice on `private` | Dice weighs the rare class by overlap, not pixel count |
| whole subsets held out for the inference examples | frames within a city are near-duplicates, so a random split would leak |

Constraint from MiTA-Segmenter's attention: **every encoder input must be exactly 32×32 (or 40×40) patches + 1 CLS token**,
so the input side is 32 × patch size (512 px for patch-16), and distilled DeiT backbones (a second extra token) cannot be used.


## 1. Config: everything that defines the run

In [ ]:
CFG = dict(
    # model: the one setting to change. Any non-distilled ViT in MiTA-Segmenter's segm/config.yml (list in the intro);
    # everything that depends on it is derived in section 1b.
    backbone="vit_tiny_patch16_384",

    # data
    data_root="/data/pp4av",
    held_out_subsets=["switzerland", "strasbourg"],   # never trained on; used for the inference examples
    px_per_patch=8,              # original-image pixels per model patch; sets the upscale factor (2x for patch-16)
    scale_jitter=(0.75, 1.25),   # random multiplier on the upscale factor during training
    object_crop_prob=0.5,        # share of training crops placed around a random object
    grid=32,                     # patches per side of every model input; MiTA attention only accepts 32 or 40

    # model settings that do not depend on the backbone (MiTA-Segmenter train.py defaults)
    n_cls=2, dropout=0.0, drop_path_rate=0.1,

    # optimisation: MiTA-Segmenter's recipe (train.py defaults + ADE20K dataset settings)
    opt="sgd", lr=0.001, momentum=0.9, weight_decay=0.0,
    sched="polynomial", poly_power=0.9, poly_step_size=1, min_lr=1e-5,
    epochs=64, batch_size=8,
    # deviations from the recipe
    dice_weight=1.0,             # loss = CE + dice_weight * Dice(private)
    amp_bf16=True,               # MiTA attention runs fp16 internally; bf16 for the rest saves time and memory

    # runtime
    device="cuda",
    num_workers=12,
    log_every=20,                # steps between log records
    plot_every=200,              # steps between live plot refreshes
    run_root="/workspace/runs",  # each backbone gets its own run directory below this

    # inference: sliding windows of the model input size over the upscaled image (MiTA recipe: stride = window)
    threshold=0.5,               # P(private) above which a pixel is masked
    mask_dilate_px=3,            # grow the mask slightly before blurring
    blur_sigma=8,
)
assert CFG["grid"] in (32, 40), "MiTA-Segmenter attention only accepts 32x32 or 40x40 patch grids"


In [ ]:
import os, time, zipfile, random, math
from pathlib import Path
from types import SimpleNamespace

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display

from segm.config import load_config
from segm.model.factory import create_segmenter
from segm.model.utils import sliding_window, merge_windows
from segm.optim.factory import create_optimizer, create_scheduler

cv2.setNumThreads(0)                     # let DataLoader workers parallelise, not OpenCV threads
DEVICE = torch.device(CFG["device"])
DATA_ROOT = Path(CFG["data_root"])
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

## 1b. Settings derived from the backbone

Encoder shape, patch size and normalisation come from MiTA-Segmenter's `segm/config.yml`, the same place its `train.py`
reads them. The input size, upscale factor and run directory follow from those. Nothing here needs editing.

In [ ]:
SEGM_CFG = load_config()
usable = [name for name, m in SEGM_CFG["model"].items() if not m.get("distilled", False)]
if CFG["backbone"] not in usable:
    why = (" (distilled DeiT models add a second extra token, which MiTA attention does not accept)"
           if CFG["backbone"] in SEGM_CFG["model"] else "")
    raise ValueError(f"backbone {CFG['backbone']!r} cannot be used{why}. Choose one of: {', '.join(usable)}")
if CFG["backbone"] == "vit_base_patch8_384":
    # MiTA-Segmenter loads this one from a local file instead of downloading it
    weights = Path(os.path.expandvars("$TORCH_HOME/hub/checkpoints/vit_base_patch8_384.pth"))
    if not weights.exists():
        raise FileNotFoundError(f"vit_base_patch8_384 needs its ImageNet weights at {weights} (set TORCH_HOME if it is unset)")

enc = SEGM_CFG["model"][CFG["backbone"]]
CFG["model"] = dict(
    backbone=CFG["backbone"], patch_size=enc["patch_size"], d_model=enc["d_model"], n_heads=enc["n_heads"],
    n_layers=enc["n_layers"], normalization=enc["normalization"], distilled=False,
    dropout=CFG["dropout"], drop_path_rate=CFG["drop_path_rate"], n_cls=CFG["n_cls"],
    decoder=dict(SEGM_CFG["decoder"]["mask_transformer"], name="mask_transformer"),
)
CFG["crop"] = CFG["window"] = CFG["window_stride"] = CFG["grid"] * enc["patch_size"]   # model input side in pixels
CFG["upscale"] = enc["patch_size"] / CFG["px_per_patch"]
CFG["run_dir"] = str(Path(CFG["run_root"]) / f"mita_{CFG['backbone']}_pp4av")
RUN_DIR = Path(CFG["run_dir"]); RUN_DIR.mkdir(parents=True, exist_ok=True)

NORM_STATS = {   # segm/data/utils.py STATS (importing segm.data would pull in its dataset dependencies)
    "vit": dict(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    "deit": dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
}

pd.Series({
    "backbone": CFG["backbone"],
    "encoder": f"patch {enc['patch_size']}, width {enc['d_model']}, {enc['n_heads']} heads, {enc['n_layers']} layers",
    "normalisation": enc["normalization"],
    "model input": f"{CFG['crop']}x{CFG['crop']} px ({CFG['grid']}x{CFG['grid']} patches)",
    "upscale": f"{CFG['upscale']:g}x (one patch = {CFG['px_per_patch']} original px)",
    "run dir": CFG["run_dir"],
}).to_frame("derived")


## 2. Data: index PP4AV and split by subset

In [ ]:
def fetch_and_extract(zip_name, out_name, repo="khaclinh/pp4av"):
    out_dir = DATA_ROOT / out_name
    if out_dir.exists() and any(out_dir.iterdir()):
        return out_dir
    zip_path = DATA_ROOT / "raw" / "data" / zip_name
    if not zip_path.exists():
        from huggingface_hub import hf_hub_download
        hf_hub_download(repo, f"data/{zip_name}", repo_type="dataset", local_dir=DATA_ROOT / "raw")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    return out_dir

IMG_DIR = fetch_and_extract("images.zip", "images")
ANN_DIR = fetch_and_extract("annotations.zip", "annotations")

def load_index():
    """One record per image: path, subset, and its boxes as an (N, 4) float array of pixel corners
    x1, y1, x2, y2. Label files store fractions of the image size; faces and plates are merged into one class."""
    from PIL import Image
    labels = {(p.parent.name, p.stem): p for p in ANN_DIR.rglob("*.txt")}
    items = []
    for p in sorted(q for q in IMG_DIR.rglob("*") if q.suffix.lower() in {".png", ".jpg", ".jpeg"}):
        subset = p.parent.name
        with Image.open(p) as im:          # reads the header only
            W, H = im.size
        boxes = []
        ann = labels.get((subset, p.stem))
        if ann is not None:
            for line in ann.read_text().splitlines():
                parts = line.split()
                if len(parts) >= 5:
                    cx, cy, bw, bh = map(float, parts[1:5])
                    boxes.append(((cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H))
        items.append(dict(path=str(p), subset=subset, boxes=np.array(boxes, np.float32).reshape(-1, 4)))
    return items

index = load_index()
train_items = [it for it in index if it["subset"] not in CFG["held_out_subsets"]]
held_items = [it for it in index if it["subset"] in CFG["held_out_subsets"]]
print(f"train: {len(train_items)} images, {sum(len(i['boxes']) for i in train_items)} objects")
print(f"held out ({', '.join(CFG['held_out_subsets'])}): {len(held_items)} images, {sum(len(i['boxes']) for i in held_items)} objects")

### Training samples

Each sample: cut a square window from the original image, sized so that resizing it to 512 px gives the chosen scale
(`CFG['upscale']`, jittered), then rasterise the boxes inside it as the `private` mask. With probability `object_crop_prob`
the window is placed so it contains a random object. Random horizontal flip and brightness/contrast jitter follow.

In [ ]:
class CropDataset(torch.utils.data.Dataset):
    def __init__(self, items, cfg):
        self.items, self.cfg = items, cfg

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        cfg, it = self.cfg, self.items[i]
        img = cv2.cvtColor(cv2.imread(it["path"], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        scale = cfg["upscale"] * random.uniform(*cfg["scale_jitter"])
        win = min(int(round(cfg["crop"] / scale)), H, W)          # window side in original pixels

        boxes = it["boxes"]
        if len(boxes) and random.random() < cfg["object_crop_prob"]:
            x1, y1, x2, y2 = boxes[random.randrange(len(boxes))]
            x0 = random.uniform(max(0, x2 - win), max(0, min(x1, W - win)))
            y0 = random.uniform(max(0, y2 - win), max(0, min(y1, H - win)))
        else:
            x0, y0 = random.uniform(0, W - win), random.uniform(0, H - win)
        x0, y0 = int(min(x0, W - win)), int(min(y0, H - win))

        crop = cv2.resize(img[y0:y0 + win, x0:x0 + win], (cfg["crop"], cfg["crop"]), interpolation=cv2.INTER_LINEAR)
        mask = np.zeros((cfg["crop"], cfg["crop"]), np.uint8)
        s = cfg["crop"] / win
        for bx1, by1, bx2, by2 in boxes:
            u1, v1 = int(np.floor((bx1 - x0) * s)), int(np.floor((by1 - y0) * s))
            u2, v2 = int(np.ceil((bx2 - x0) * s)), int(np.ceil((by2 - y0) * s))
            if u2 > 0 and v2 > 0 and u1 < cfg["crop"] and v1 < cfg["crop"]:
                mask[max(v1, 0):v2, max(u1, 0):u2] = 1

        if random.random() < 0.5:
            crop, mask = crop[:, ::-1], mask[:, ::-1]
        alpha, beta = random.uniform(0.8, 1.2), random.uniform(-20, 20)
        crop = np.clip(crop.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

        return torch.from_numpy(np.ascontiguousarray(crop)).permute(2, 0, 1), torch.from_numpy(np.ascontiguousarray(mask)).long()

_MEAN = torch.tensor(NORM_STATS[CFG["model"]["normalization"]]["mean"], device=DEVICE).view(1, 3, 1, 1)
_STD = torch.tensor(NORM_STATS[CFG["model"]["normalization"]]["std"], device=DEVICE).view(1, 3, 1, 1)

def normalize(x_uint8):
    """uint8 [B,3,H,W] on device -> the backbone's normalisation (`normalization` in segm/config.yml)."""
    return (x_uint8.float() / 255.0 - _MEAN) / _STD

train_ds = CropDataset(train_items, CFG)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=CFG["batch_size"], shuffle=True, drop_last=True,
    num_workers=CFG["num_workers"], pin_memory=True, persistent_workers=CFG["num_workers"] > 0,
)
print(f"{len(train_loader)} steps per epoch, {len(train_loader) * CFG['epochs']} steps total")

In [ ]:
# sanity check: training crops with their masks (red)
ims, masks = next(iter(torch.utils.data.DataLoader(train_ds, batch_size=8, shuffle=True)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, im, m in zip(axes.flat, ims, masks):
    ax.imshow(im.permute(1, 2, 0).numpy())
    ax.imshow(np.ma.masked_where(m.numpy() == 0, m.numpy()), cmap="autumn", alpha=0.6, vmin=0, vmax=1)
    ax.set_title(f"private pixels: {100 * m.float().mean():.2f}%", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

### Patch: MiTA's decoder assumes ADE20K's 150 classes

MiTA-Segmenter's decoder attention has two ADE20K numbers written in as literals: `150` (class tokens) and
`300` (its internal width, 3 heads × 100). Below, the same forward pass with those literals replaced by the values they
stand for. At 150 classes it is bit-identical to MiTA's original (checked in the cell).

In [ ]:
import segm.model.dec_blocks as dec_blocks
from einops import rearrange

_mita_decoder_forward = dec_blocks.Attention.forward

def _decoder_forward_any_n_cls(self, x):
    b, n, c = x.shape
    n_cls, inner = self.n_cls, self.heads * self.dim_head          # MiTA: 150, 300
    h = width = int((n - n_cls) ** 0.5)
    w = self.proj(x)
    rep = self.pool(w[:, :-n_cls, :].reshape(b, h, width, inner).permute(0, 3, 1, 2)).reshape(b, inner, -1).permute(0, 2, 1)
    w = w.reshape(b, n, self.heads, self.dim_head).permute(0, 2, 1, 3)
    rep = rep.reshape(b, rep.shape[1], self.heads, self.dim_head).permute(0, 2, 1, 3)
    rep_delta, attn = self.attention(rep, w, w)
    rep = rep + self.step_rep * rep_delta
    x_delta, _ = self.attention(rep, rep, rep)
    x_delta = attn.transpose(-1, -2) @ x_delta
    x_delta = self.step_x * x_delta
    return self.to_out(rearrange(x_delta, "b h n k -> b n (h k)"))

# check: identical to MiTA's original at 150 classes, at this backbone's width
# (dim_head=100 is fixed inside segm.model.decoder.MaskTransformer)
_d, _heads = CFG["model"]["d_model"], CFG["model"]["decoder"]["n_heads"]
_att = dec_blocks.Attention(_d, heads=_heads, dim_head=100).eval(); _att.n_cls = 150
_x = torch.randn(2, CFG["grid"] ** 2 + 150, _d)
with torch.no_grad():
    assert torch.equal(_mita_decoder_forward(_att, _x), _decoder_forward_any_n_cls(_att, _x))
dec_blocks.Attention.forward = _decoder_forward_any_n_cls
print("decoder patch applied (verified identical to MiTA's original at 150 classes)")

## 3. Model, optimiser, scheduler (MiTA-Segmenter's own factories)

In [ ]:
model_cfg = dict(CFG["model"], image_size=(CFG["crop"], CFG["crop"]), decoder=dict(CFG["model"]["decoder"]))
model = create_segmenter(model_cfg).to(DEVICE)
for m in model.decoder.modules():                     # tell the patched decoder attention the class count
    if isinstance(m, dec_blocks.Attention):
        m.n_cls = CFG["model"]["n_cls"]
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

opt_args = SimpleNamespace(
    opt=CFG["opt"], lr=CFG["lr"], weight_decay=CFG["weight_decay"], momentum=CFG["momentum"],
    sched=CFG["sched"], poly_power=CFG["poly_power"], poly_step_size=CFG["poly_step_size"], min_lr=CFG["min_lr"],
    epochs=CFG["epochs"], iter_max=len(train_loader) * CFG["epochs"], iter_warmup=0.0,
)
optimizer = create_optimizer(opt_args, model)
lr_scheduler = create_scheduler(opt_args, optimizer)

state = dict(step=0, epoch=0)
log = []          # one dict per `log_every` steps; turned into a DataFrame for plots and the CSV

## 4. Train

Runs in the foreground. Interrupt the kernel (■) to stop early: the checkpoint and log are saved either way,
and re-running this cell continues from where it stopped.

The plot refreshes every `plot_every` steps:
- **loss**: total, cross-entropy and Dice (smoothed)
- **learning rate**
- **throughput**: images/s, and the share of each step spent waiting for data (high = CPU-bound data loading)

In [ ]:
def dice_loss(logits, target):
    p = logits.softmax(1)[:, 1]
    t = (target == 1).float()
    inter = (p * t).sum()
    return 1 - (2 * inter + 1) / (p.sum() + t.sum() + 1)

def plot_log(log, smooth=10):
    df = pd.DataFrame(log)
    fig, axes = plt.subplots(1, 3, figsize=(17, 3.6))
    for col, c in [("loss", "k"), ("ce", "tab:blue"), ("dice", "tab:orange")]:
        axes[0].plot(df.step, df[col].rolling(smooth, min_periods=1).mean(), color=c, label=col)
    axes[0].set_yscale("log"); axes[0].set_title("loss (smoothed)"); axes[0].legend(); axes[0].set_xlabel("step")
    axes[1].plot(df.step, df.lr); axes[1].set_title("learning rate"); axes[1].set_xlabel("step")
    axes[2].plot(df.step, df.img_per_s.rolling(smooth, min_periods=1).mean(), color="tab:green")
    axes[2].set_ylabel("images / s", color="tab:green"); axes[2].set_xlabel("step")
    ax2 = axes[2].twinx()
    ax2.plot(df.step, 100 * df.data_wait.rolling(smooth, min_periods=1).mean(), color="tab:red")
    ax2.set_ylabel("% of step waiting for data", color="tab:red"); ax2.set_ylim(0, 100)
    last = df.iloc[-1]
    axes[2].set_title(f"throughput  (GPU mem peak {last.gpu_mem_gb:.1f} GB)")
    plt.tight_layout()
    return fig

def save_checkpoint():
    torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                    lr_scheduler=lr_scheduler.state_dict(), state=state, cfg=CFG), RUN_DIR / "checkpoint.pth")
    if log:
        pd.DataFrame(log).to_csv(RUN_DIR / "log.csv", index=False)

status = display("starting...", display_id=True)
plot_handle = display(display_id=True)
amp = torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=CFG["amp_bf16"])
acc = dict(loss=0.0, ce=0.0, dice=0.0, data=0.0, compute=0.0, n=0)

model.train()
try:
    while state["epoch"] < CFG["epochs"]:
        t_end = time.perf_counter()
        for ims, masks in train_loader:
            t_data = time.perf_counter()
            x = normalize(ims.to(DEVICE, non_blocking=True))
            y = masks.to(DEVICE, non_blocking=True)
            with amp:
                logits = model(x)
            logits = logits.float()
            ce = F.cross_entropy(logits, y)
            dice = dice_loss(logits, y)
            loss = ce + CFG["dice_weight"] * dice

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            state["step"] += 1
            lr_scheduler.step_update(num_updates=state["step"])
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_step = time.perf_counter()

            acc["loss"] += loss.item(); acc["ce"] += ce.item(); acc["dice"] += dice.item()
            acc["data"] += t_data - t_end; acc["compute"] += t_step - t_data; acc["n"] += 1
            t_end = t_step

            if state["step"] % CFG["log_every"] == 0:
                n, total_t = acc["n"], acc["data"] + acc["compute"]
                log.append(dict(
                    step=state["step"], epoch=state["epoch"],
                    loss=acc["loss"] / n, ce=acc["ce"] / n, dice=acc["dice"] / n,
                    lr=optimizer.param_groups[0]["lr"],
                    img_per_s=n * CFG["batch_size"] / total_t, step_time_ms=1000 * total_t / n,
                    data_wait=acc["data"] / total_t,
                    gpu_mem_gb=torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0,
                ))
                acc = dict(loss=0.0, ce=0.0, dice=0.0, data=0.0, compute=0.0, n=0)
                r = log[-1]
                status.update(f"epoch {r['epoch'] + 1}/{CFG['epochs']}  step {r['step']}/{len(train_loader) * CFG['epochs']}  "
                              f"loss {r['loss']:.4f}  {r['img_per_s']:.0f} img/s  data wait {100 * r['data_wait']:.0f}%")
            if state["step"] % CFG["plot_every"] == 0 and log:
                fig = plot_log(log); plot_handle.update(fig); plt.close(fig)
        state["epoch"] += 1
        save_checkpoint()
except KeyboardInterrupt:
    print("interrupted")
finally:
    save_checkpoint()
    print(f"saved {RUN_DIR / 'checkpoint.pth'} at epoch {state['epoch']}, step {state['step']}")
    if log:
        fig = plot_log(log); plot_handle.update(fig); plt.close(fig)

In [ ]:
# the full log as a table (also saved as log.csv in the run directory)
pd.DataFrame(log).tail(10)

## 5. Inference examples on held-out cities

Full-resolution inference: upscale the image by `CFG['upscale']`, run the model on sliding windows of the model input size, merge, and map the
`private` probability back to the original resolution. Pixels above `threshold` are blurred.

To run this section without retraining, run sections 1–3 and then the next cell, which loads the saved checkpoint.

In [ ]:
ckpt = torch.load(RUN_DIR / "checkpoint.pth", map_location=DEVICE, weights_only=False)
if ckpt["cfg"].get("backbone") != CFG["backbone"]:
    raise ValueError(f"{RUN_DIR / 'checkpoint.pth'} was trained with {ckpt['cfg'].get('backbone')}, not {CFG['backbone']}")
model.load_state_dict(ckpt["model"])
model.eval()
print(f"loaded checkpoint from epoch {ckpt['state']['epoch']}, step {ckpt['state']['step']}")

@torch.no_grad()
def predict(img_rgb, batch=16):
    """RGB uint8 [H, W, 3] -> P(private) as float32 [H, W] at the original resolution."""
    H, W = img_rgb.shape[:2]
    x = normalize(torch.from_numpy(img_rgb).permute(2, 0, 1)[None].to(DEVICE))
    x = F.interpolate(x, scale_factor=CFG["upscale"], mode="bilinear", align_corners=False)
    windows = sliding_window(x, False, CFG["window"], CFG["window_stride"])
    crops = torch.cat(windows.pop("crop"))
    out = []
    with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=CFG["amp_bf16"]):
        for i in range(0, len(crops), batch):
            out.append(model(crops[i:i + batch]).float())
    windows["seg_maps"] = torch.cat(out)
    prob = merge_windows(windows, CFG["window"], (H, W))      # softmax over classes, resized to (H, W)
    return prob[1].float().cpu().numpy()

def anonymize(img_rgb, prob):
    mask = (prob > CFG["threshold"]).astype(np.uint8)
    if CFG["mask_dilate_px"] > 0:
        k = 2 * CFG["mask_dilate_px"] + 1
        mask = cv2.dilate(mask, np.ones((k, k), np.uint8))
    blurred = cv2.GaussianBlur(img_rgb, (0, 0), CFG["blur_sigma"])
    return np.where(mask[..., None] == 1, blurred, img_rgb), mask

In [ ]:
# full images: the held-out images with the most objects
examples = sorted(held_items, key=lambda it: -len(it["boxes"]))[:4]
results = []
for it in examples:
    img = cv2.cvtColor(cv2.imread(it["path"]), cv2.COLOR_BGR2RGB)
    t = time.perf_counter(); prob = predict(img); dt = time.perf_counter() - t
    anon, mask = anonymize(img, prob)
    results.append((it, img, prob, anon, mask))

    fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))
    axes[0].imshow(img)
    for x1, y1, x2, y2 in it["boxes"]:
        axes[0].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", lw=1.5))
    axes[0].set_title(f"{it['subset']}: input + labelled boxes (green)")
    axes[1].imshow(prob, cmap="magma", vmin=0, vmax=1); axes[1].set_title(f"P(private)   ({dt * 1000:.0f} ms)")
    axes[2].imshow(anon); axes[2].set_title("anonymized")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
# close-ups: labelled objects in the held-out images, original vs. anonymized
pool = [(it, img, anon, b) for it, img, prob, anon, mask in results for b in it["boxes"]]
random.Random(0).shuffle(pool)
pool = pool[:12]
if pool:
    fig, axes = plt.subplots(2, len(pool), figsize=(1.6 * len(pool), 3.6), squeeze=False)
    for k, (it, img, anon, (x1, y1, x2, y2)) in enumerate(pool):
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        r = int(max(x2 - x1, y2 - y1) * 1.5 + 8)
        sl = (slice(max(0, int(cy) - r), int(cy) + r), slice(max(0, int(cx) - r), int(cx) + r))
        axes[0, k].imshow(img[sl], interpolation="nearest"); axes[1, k].imshow(anon[sl], interpolation="nearest")
        axes[0, k].set_title(f"{x2 - x1:.0f}x{y2 - y1:.0f}px", fontsize=8)
        axes[0, k].axis("off"); axes[1, k].axis("off")
    axes[0, 0].text(-0.15, 0.5, "original", transform=axes[0, 0].transAxes, rotation=90, va="center", ha="right")
    axes[1, 0].text(-0.15, 0.5, "anonymized", transform=axes[1, 0].transAxes, rotation=90, va="center", ha="right")
    plt.tight_layout(); plt.show()